# 🤖 Módulo 1 · Cuaderno 1 — Nexo aprende a hablar
### Fundamentos de los modelos de lenguaje y anatomía del prompt

**AVE Ingeniería de Prompting y Agentes Autónomos · Universidad Ean**

La Cooperativa Raíces probó un chatbot genérico: Don Aurelio le preguntó tres veces por el precio de referencia del café y recibió **tres respuestas distintas; una era inventada**. En este cuaderno vas a descubrir por qué pasa eso y vas a dar los primeros pasos para que **Nexo**, el asistente de la cooperativa, responda bien.

**Al terminar podrás:**
1. Conectarte a un modelo de lenguaje (LLM) desde Python.
2. Explicar con ejemplos por qué la misma IA responde distinto (temperatura y muestreo).
3. Medir **tokens** y entender la **ventana de contexto**.
4. Ver qué son los **embeddings** y cómo miden el parecido entre frases.
5. Escribir un prompt con sus cinco partes: **rol, contexto, instrucción, formato y ejemplos**.

> Relación con el AVE: Módulo 1, Págs. 01 a 05.

## 0. Antes de empezar: cómo usar este cuaderno

Un **cuaderno de Colab** mezcla texto (como este) y **celdas de código** que puedes ejecutar.

- Para ejecutar una celda, haz clic en ella y presiona **Shift + Enter** (o el botón ▶ de la izquierda).
- Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla, vuelve a ejecutar desde la sección de configuración.
- Lo que escribe el código aparece debajo de la celda: eso es la **salida**.
- Si cierras la pestaña o pasa mucho tiempo, Colab reinicia la sesión y hay que ejecutar todo otra vez (*Entorno de ejecución → Ejecutar todas*).

### Tu clave de Gemini (gratis)

Nexo usa los modelos **Gemini** de Google a través de su API. Necesitas una clave personal:

1. Entra a <https://aistudio.google.com/apikey> con tu cuenta de Google y haz clic en **Create API key**.
2. Copia la clave.
3. En Colab, abre el panel izquierdo y haz clic en el ícono de la **llave 🔑 (Secretos)**.
4. Agrega un secreto con nombre `GOOGLE_API_KEY`, pega la clave en *Valor* y activa **Acceso al cuaderno**.

> ⚠️ **Nunca escribas tu clave dentro del código** ni la compartas: si entregas o compartes el cuaderno, cualquiera podría usarla. Los *Secretos* de Colab la guardan aparte.

La capa gratuita de Gemini tiene un límite de solicitudes por minuto. Si ves un error **429**, espera un minuto: el código de este cuaderno ya reintenta solo.

## 1. Configuración
Ejecuta las dos celdas siguientes. La primera instala las librerías; la segunda conecta con Gemini y prueba que todo funcione.

In [ ]:
# ── Instalación de librerías ─────────────────────────────────────────────
# El signo "!" ejecuta un comando de la terminal dentro de Colab.
# google-genai : la librería oficial de Google para usar los modelos Gemini.
# pydantic     : sirve para definir "moldes" de datos y pedir respuestas con estructura fija.
# -q (quiet)   : instala sin llenar la pantalla de mensajes.
!pip install -q -U google-genai pydantic

In [ ]:
# ── Configuración: conectar con Gemini ───────────────────────────────────
import time                      # para medir tiempos y esperar entre reintentos
import json                      # para leer y escribir datos en formato JSON
from google import genai         # cliente para hablar con los modelos Gemini
from google.genai import types   # "tipos": objetos de configuración (temperatura, herramientas, etc.)


def obtener_clave():
    """Lee la clave desde los Secretos de Colab; si no existe, la pide por teclado."""
    try:
        from google.colab import userdata
        clave = userdata.get("GOOGLE_API_KEY")
        if clave:
            return clave
    except Exception:
        pass  # no estamos en Colab o el secreto no tiene acceso al cuaderno
    from getpass import getpass  # getpass oculta lo que escribes
    return getpass("Pega tu clave de Gemini y presiona Enter: ")


# El "cliente" es el objeto que envía nuestras preguntas al modelo y trae las respuestas.
client = genai.Client(api_key=obtener_clave())

# Los nombres de los modelos cambian con el tiempo. Probamos una lista en orden de preferencia
# y nos quedamos con el primero que esté disponible para tu clave.
MODELOS_PREFERIDOS = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.0-flash"]


def elegir_modelo(preferidos=MODELOS_PREFERIDOS):
    disponibles = []
    for m in client.models.list():
        acciones = m.supported_actions or []
        if "generateContent" in acciones:
            disponibles.append(m.name.replace("models/", ""))
    for nombre in preferidos:
        if nombre in disponibles:
            return nombre
    # Si ninguno de la lista existe, usamos cualquier modelo "flash" disponible.
    flash = [n for n in disponibles if "flash" in n and "image" not in n and "tts" not in n]
    return flash[0] if flash else disponibles[0]


MODELO = elegir_modelo()
print("✅ Conectado. Modelo que usará Nexo:", MODELO)


def llamar_con_reintentos(funcion, *args, intentos=5, **kwargs):
    """Ejecuta una llamada a la API y, si el servicio está saturado (error 429 o 503),
    espera y vuelve a intentar. Así el cuaderno no se detiene por límites de la capa gratuita."""
    espera = 10
    for intento in range(1, intentos + 1):
        try:
            return funcion(*args, **kwargs)
        except Exception as error:
            mensaje = str(error)
            if any(codigo in mensaje for codigo in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")) and intento < intentos:
                print(f"⏳ Servicio ocupado (intento {intento}). Espero {espera} s y reintento...")
                time.sleep(espera)
                espera *= 2  # cada vez esperamos el doble
            else:
                raise


def preguntar(prompt, sistema=None, temperatura=None, modelo=None, **otros):
    """Envía un texto al modelo y devuelve la respuesta como texto.
    - prompt:      lo que le pedimos al modelo.
    - sistema:     instrucciones de sistema (quién es y cómo se comporta Nexo).
    - temperatura: 0 = respuestas más estables; 1 o más = respuestas más variadas.
    - otros:       cualquier otro ajuste de GenerateContentConfig."""
    config = types.GenerateContentConfig(system_instruction=sistema, temperature=temperatura, **otros)
    respuesta = llamar_con_reintentos(
        client.models.generate_content, model=modelo or MODELO, contents=prompt, config=config
    )
    return respuesta.text


print(preguntar("Saluda en una sola frase, en español."))

## 2. Los datos de la cooperativa

Un modelo de lenguaje **no sabe** nada de la Cooperativa Raíces: aprendió de textos de internet, no de nuestros archivos. Por eso guardamos aquí, en variables de Python, la información oficial. Más adelante se la daremos a Nexo como **contexto**.

- Un **diccionario** (`{clave: valor}`) guarda datos con nombre, como el precio.
- Una **lista** (`[a, b, c]`) guarda varios elementos en orden, como los artículos del reglamento.

In [ ]:
# ── Datos (ficticios) de la Cooperativa Raíces ───────────────────────────
# Son la "fuente de verdad" de la cooperativa. Nexo no debe inventar nada que no esté aquí.
COOPERATIVA = {
    "nombre": "Cooperativa Raíces",
    "producto": "café pergamino seco",
    "precio_referencia_carga": 2_150_000,   # pesos colombianos por carga de 125 kg (se actualiza cada lunes)
    "fecha_precio": "lunes de esta semana",
    "horario_acopio": "lunes a sábado, de 6:00 a. m. a 2:00 p. m.",
}

REGLAMENTO = [
    "Art. 1. Precio de referencia: la cooperativa publica cada lunes el precio de referencia por carga de 125 kg de café pergamino seco. Nexo solo informa el precio vigente publicado.",
    "Art. 2. Calidad: el café se recibe con humedad entre 10 % y 12 %. Si la humedad supera el 12 %, se descuenta el 1 % del valor por cada punto adicional.",
    "Art. 3. Horario del centro de acopio: lunes a sábado de 6:00 a. m. a 2:00 p. m. Los domingos y festivos no hay recepción.",
    "Art. 4. Recogida en camión: solo para entregas de 10 cargas o más, en veredas con vía carreteable, y se solicita con 48 horas de anticipación.",
    "Art. 5. Pagos: los pagos se hacen por transferencia a la cuenta registrada del asociado dentro de los 3 días hábiles siguientes a la entrega.",
    "Art. 6. Aprobación de pagos: todo pago igual o superior a 5.000.000 de pesos requiere la aprobación de la analista de pagos.",
    "Art. 7. Anticipos de cosecha: un asociado puede pedir un anticipo de hasta el 30 % del valor entregado el año anterior.",
    "Art. 8. Datos personales: Nexo no comparte datos de un asociado con otra persona y nunca pide contraseñas ni claves bancarias.",
    "Art. 9. Quejas: toda queja se registra con un número de caso y la responde una persona del equipo en máximo 5 días hábiles.",
    "Art. 10. Asamblea: la asamblea general de asociados se realiza el último sábado de marzo.",
]

ASOCIADOS = {
    "A-101": {"nombre": "Aurelio Gómez", "vereda": "La Esperanza", "via_carreteable": True,  "cargas_anio_anterior": 48},
    "A-102": {"nombre": "Rosa Pineda",   "vereda": "El Diviso",    "via_carreteable": False, "cargas_anio_anterior": 22},
    "A-103": {"nombre": "Julián Ortiz",  "vereda": "Alto Bonito",  "via_carreteable": True,  "cargas_anio_anterior": 9},
}

print("Precio de referencia:", f"${COOPERATIVA['precio_referencia_carga']:,}".replace(",", "."), "por carga")
print("Artículos del reglamento:", len(REGLAMENTO), "| Asociados registrados:", len(ASOCIADOS))

## 3. ¿Por qué la misma IA responde distinto?

Un LLM escribe **una palabra (token) a la vez**. En cada paso calcula la probabilidad de muchos tokens posibles y **elige uno al azar** según esas probabilidades. Ese sorteo se controla con la **temperatura**:

| Temperatura | Qué pasa |
|---|---|
| 0 | Casi siempre elige el token más probable → respuestas estables. |
| 1 | Sorteo "normal" → respuestas variadas. |
| 2 | Sorteo muy abierto → respuestas creativas, a veces incoherentes. |

Vamos a repetir el experimento de Don Aurelio: la misma pregunta, tres veces, **sin darle datos** al modelo.

In [ ]:
pregunta = "¿Cuál es el precio de referencia del café en la Cooperativa Raíces esta semana? Responde con una cifra."

# Repetimos la misma pregunta 3 veces con temperatura alta (1.5)
for intento in range(1, 4):
    respuesta = preguntar(pregunta, temperatura=1.5)
    print(f"Intento {intento}: {respuesta.strip()}\n")

**¿Qué observas?** Lo más probable es que las respuestas cambien y que alguna invente una cifra con total seguridad. Eso es una **alucinación**: el modelo produce un texto plausible, no un dato verificado.

Ahora probemos con **temperatura 0**. ¿Se arregla el problema?

In [ ]:
for intento in range(1, 4):
    respuesta = preguntar(pregunta, temperatura=0)
    print(f"Intento {intento}: {respuesta.strip()}\n")

Con temperatura 0 las respuestas se parecen más entre sí, pero **pueden seguir siendo inventadas**: ser consistente no es lo mismo que ser correcto. La solución de fondo es **darle al modelo la información verdadera** (contexto) y pedirle que no invente.

In [ ]:
contexto = f"""Datos oficiales de {COOPERATIVA['nombre']}:
- Precio de referencia: {COOPERATIVA['precio_referencia_carga']} pesos por carga de 125 kg de {COOPERATIVA['producto']}.
- Fecha de publicación: {COOPERATIVA['fecha_precio']}."""

prompt_con_datos = f"""{contexto}

Usa SOLO los datos anteriores. Si la respuesta no está en los datos, di "No tengo ese dato".
Pregunta: {pregunta}"""

for intento in range(1, 4):
    print(f"Intento {intento}: {preguntar(prompt_con_datos, temperatura=1.5).strip()}\n")

✅ Aun con temperatura alta, la cifra ya no cambia: el modelo **copia el dato del contexto** en vez de imaginarlo. Esta idea, darle al modelo información confiable, es la base de todo lo que construiremos (en el Módulo 2 se llama **RAG**).

> **Para tu bitácora:** anota con tus palabras por qué el primer chatbot de la cooperativa falló y qué cambió en el último experimento.

## 4. Tokens y ventana de contexto

Los modelos no leen letras ni palabras completas: leen **tokens**, trozos de texto (una palabra corta, parte de una palabra larga, un signo). Todo tiene un límite:

- La **ventana de contexto** es el máximo de tokens que el modelo puede "tener en mente" a la vez (pregunta + contexto + respuesta).
- El **costo** y el **tiempo** de respuesta crecen con la cantidad de tokens.

Contemos tokens con la función `count_tokens`:

In [ ]:
textos = {
    "Saludo corto": "Hola, Nexo",
    "Pregunta de Don Aurelio": pregunta,
    "Reglamento completo": "\n".join(REGLAMENTO),
}

for nombre, texto in textos.items():
    conteo = llamar_con_reintentos(client.models.count_tokens, model=MODELO, contents=texto)
    palabras = len(texto.split())
    print(f"{nombre:<25} palabras: {palabras:>4}   tokens: {conteo.total_tokens:>4}")

In [ ]:
# ¿Qué tan grande es la ventana de contexto del modelo que estamos usando?
info = client.models.get(model=MODELO)
print("Modelo:", info.name)
print("Tokens de entrada máximos (ventana de contexto):", f"{info.input_token_limit:,}".replace(",", "."))
print("Tokens de salida máximos:", f"{info.output_token_limit:,}".replace(",", "."))

La ventana es enorme, pero eso no significa que convenga llenarla: los modelos atienden peor la información que queda **en el medio** de un contexto muy largo (Liu et al., 2024). Por eso, más adelante le daremos a Nexo **solo los fragmentos relevantes**.

## 5. Embeddings: el significado convertido en números

Dentro del Transformer, cada texto se representa como un **vector**: una lista larga de números llamada **embedding**. Textos con significado parecido quedan con vectores parecidos, aunque usen palabras distintas.

Para comparar dos vectores usamos la **similitud coseno**: 1 = mismo sentido, 0 = nada que ver.

In [ ]:
import numpy as np      # numpy: cálculos con vectores y matrices
import pandas as pd     # pandas: tablas de datos

frases = [
    "¿A cómo están pagando la carga de café?",
    "¿Cuál es el precio de referencia esta semana?",
    "¿Pueden recoger mi cosecha en camión?",
    "Necesito que el camión pase por mi finca",
    "¿Cuándo es la asamblea de asociados?",
]

# Pedimos un embedding por frase. MODELO_EMB es el modelo especializado en embeddings.
MODELO_EMB = "gemini-embedding-001"
resultado = llamar_con_reintentos(client.models.embed_content, model=MODELO_EMB, contents=frases)
vectores = np.array([e.values for e in resultado.embeddings])
print("Forma de la matriz:", vectores.shape, "→ (frases, números por frase)")

# Normalizamos cada vector (largo 1) para que el producto punto sea la similitud coseno.
vectores = vectores / np.linalg.norm(vectores, axis=1, keepdims=True)
similitud = vectores @ vectores.T

etiquetas = [f[:28] + "…" for f in frases]
tabla = pd.DataFrame(similitud, index=etiquetas, columns=range(1, 6)).round(2)
tabla.style.background_gradient(cmap="Blues")

Observa que las dos preguntas sobre el **precio** se parecen mucho entre sí, igual que las dos sobre el **camión**, aunque casi no compartan palabras. Esto es lo que permitirá a Nexo **buscar** en el reglamento el artículo correcto (Módulo 2).

## 6. Anatomía del prompt

Un buen prompt tiene cinco partes:

| Parte | Pregunta que responde | Ejemplo para Nexo |
|---|---|---|
| **Rol** | ¿Quién eres? | Asistente de atención de la Cooperativa Raíces |
| **Contexto** | ¿Qué sabes? | Precio vigente, reglamento |
| **Instrucción** | ¿Qué debes hacer? | Responder la consulta del asociado |
| **Formato** | ¿Cómo lo entregas? | Máximo 3 frases, tono cercano, cita el artículo |
| **Ejemplos** | ¿Cómo se ve una buena respuesta? | Una pregunta y su respuesta modelo |

Vamos a escribir una función que arme el prompt a partir de esas piezas:

In [ ]:
def construir_prompt(rol, contexto, instruccion, formato, ejemplos, consulta):
    """Arma un prompt con sus cinco partes, cada una bajo un título claro."""
    return f"""# ROL
{rol}

# CONTEXTO
{contexto}

# INSTRUCCIÓN
{instruccion}

# FORMATO DE RESPUESTA
{formato}

# EJEMPLOS
{ejemplos}

# CONSULTA DEL ASOCIADO
{consulta}"""


rol = "Eres Nexo, el asistente virtual de atención a asociados de la Cooperativa Raíces, una cooperativa cafetera."
contexto_nexo = contexto + "\nReglamento:\n" + "\n".join(REGLAMENTO)
instruccion = ("Responde la consulta usando únicamente el contexto. Si la información no está, "
               "dilo con claridad y ofrece comunicar al asociado con una persona del equipo.")
formato = "Máximo 3 frases, en español sencillo y respetuoso. Termina citando el artículo que usaste, por ejemplo (Art. 4)."
ejemplos = """Consulta: ¿Reciben café el domingo?
Respuesta: No, los domingos y festivos el centro de acopio no recibe café. Puede traerlo de lunes a sábado entre 6:00 a. m. y 2:00 p. m. (Art. 3)"""

consulta = "Buenas, tengo 6 cargas en la vereda La Esperanza, ¿me las pueden recoger en camión mañana?"
prompt_completo = construir_prompt(rol, contexto_nexo, instruccion, formato, ejemplos, consulta)
print(prompt_completo[:600], "...")

Comparemos un **prompt pobre** (solo la pregunta) con el **prompt completo**:

In [ ]:
print("❌ PROMPT POBRE\n", preguntar(consulta, temperatura=0.2))
print("\n✅ PROMPT COMPLETO\n", preguntar(prompt_completo, temperatura=0.2))

Con el prompt completo, Nexo debería explicar que la recogida en camión **no aplica** (son 6 cargas y el mínimo es 10, Art. 4). Con el prompt pobre, el modelo suele prometer cosas que el reglamento no permite: justo el error del chatbot que prometió "una recogida en camión que el reglamento no cubría".

## 7. Tipos de modelos

No todos los modelos sirven para todo. Los modelos *flash* o *lite* son rápidos y baratos; los *pro* razonan mejor pero tardan más. Veamos qué modelos tienes disponibles:

In [ ]:
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []) and "gemini" in m.name:
        print(f"{m.name.replace('models/', ''):<40} contexto: {m.input_token_limit:>10,}".replace(",", "."))

In [ ]:
# Comparamos tiempo de respuesta entre dos modelos con la misma consulta.
# Si alguno no está disponible para tu clave, cámbialo por otro nombre de la lista anterior.
for nombre in [MODELO, "gemini-flash-lite-latest"]:
    try:
        inicio = time.time()
        texto = preguntar(prompt_completo, temperatura=0.2, modelo=nombre)
        print(f"{nombre}: {time.time() - inicio:.1f} s\n{texto.strip()}\n")
    except Exception as error:
        print(f"{nombre}: no disponible ({str(error)[:80]})\n")

## 8. Reto para tu proyecto 🎯

1. Elige el caso de tu agente (real o imaginario) y escribe sus **datos oficiales** en un diccionario y una lista, como `COOPERATIVA` y `REGLAMENTO`.
2. Repite el experimento de la sección 3 con una pregunta de tu caso: sin contexto y con contexto.
3. Escribe el primer prompt de tu agente con las cinco partes usando `construir_prompt`.
4. Guarda en tu bitácora: el prompt, las respuestas y qué aprendiste.

**Lecturas de apoyo:** Shanahan (2024), *Talking about large language models*; Liu et al. (2024), *Lost in the middle*; Liu et al. (2023), *Pre-train, prompt, and predict*.

## 📄 ¿Cómo guardar este cuaderno en PDF con todas las salidas?

1. *Entorno de ejecución → Reiniciar y ejecutar todas* (o *Ejecutar todas*). Espera a que todas las celdas terminen.
2. Revisa que cada celda muestre su salida y que no haya errores en rojo.
3. *Archivo → Imprimir* y, en *Destino*, elige **Guardar como PDF**.
4. Para el `.ipynb`: *Archivo → Descargar → Descargar .ipynb*.